In [1]:
import sccellfie
import scanpy as sc
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import glasbey

import textwrap

## To avoid warnings
import warnings
warnings.filterwarnings("ignore")

/home/kvalem/.conda/envs/sccellfie/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/kvalem/.conda/envs/sccellfie/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [2]:
import anndata as ad

In [3]:
adata = ad.read_h5ad("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/002_annotate_adata.h5ad")

In [4]:
mask = (adata.obs["cell_annotation_05"] != "Cycling") & (
    adata.obs["group4"] == "ctrl"
)

# 2. Subset the AnnData object
without_cycling_ctrl= adata[mask]

In [5]:
adata = without_cycling_ctrl

In [6]:
adata = adata[~adata.obs_names.duplicated(), :]

In [7]:
import os
import glob
import scvelo as scv
import scanpy as sc

def read_scvelo_2021_looms():
    loom_dir = "/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/trajectory_inference/loom"
    
    # find all loom files starting with 2021
    loom_files = sorted(
        glob.glob(os.path.join(loom_dir, "2021*.loom"))
    )
    
    print(f"Found {len(loom_files)} loom files.")
    
    adatas = []
    
    for file in loom_files:
        print(f"Reading {os.path.basename(file)}")
        
        adata = ad.read_loom(file)
        adata.var_names_make_unique()
        
        # store filename (without extension) as metadata
        sample_name = os.path.basename(file).replace(".loom", "")
        adata.obs["sample_id"] = sample_name
        
        adatas.append(adata)
    
    # concatenate all
    adata_combined = sc.concat(adatas, join="outer", label="batch", keys=[a.obs["sample_id"][0] for a in adatas])
    
    return adata_combined


In [8]:
adata_velocity = read_scvelo_2021_looms()

Found 8 loom files.
Reading 2021_10mix-ICI1.loom
Reading 2021_10mix-ICI2.loom
Reading 2021_11mix-ICI1.loom
Reading 2021_11mix-ICI2.loom
Reading 2021_GF-ICI1-plus.loom
Reading 2021_GF-ICI1.loom
Reading 2021_GF-ICI2-plus.loom
Reading 2021_GF-ICI2.loom


In [9]:
adata_scvelo = adata_velocity

In [10]:
adata_scvelo

AnnData object with n_obs × n_vars = 72592 × 32285
    obs: 'sample_id', 'batch'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced'

In [11]:
adata.obs_names

Index(['10mix-ICI1_AAACCTGAGAGCCTAG-1', '10mix-ICI1_AAACCTGAGATGTGGC-1',
       '10mix-ICI1_AAACCTGAGCCACGTC-1', '10mix-ICI1_AAACCTGAGGCTCTTA-1',
       '10mix-ICI1_AAACCTGAGGGCACTA-1', '10mix-ICI1_AAACCTGAGGGCTTGA-1',
       '10mix-ICI1_AAACCTGAGGTTACCT-1', '10mix-ICI1_AAACCTGAGTCCAGGA-1',
       '10mix-ICI1_AAACCTGCAAAGCAAT-1', '10mix-ICI1_AAACCTGCAACCGCCA-1',
       ...
       '10mix-ICI2_TTTGTCACATTGTGCA-1', '10mix-ICI2_TTTGTCAGTACCGTTA-1',
       '10mix-ICI2_TTTGTCAGTCGAGATG-1', '10mix-ICI2_TTTGTCAGTGCGATAG-1',
       '10mix-ICI2_TTTGTCAGTGTTTGTG-1', '10mix-ICI2_TTTGTCATCACGAAGG-1',
       '10mix-ICI2_TTTGTCATCAGCTCGG-1', '10mix-ICI2_TTTGTCATCCATGCTC-1',
       '10mix-ICI2_TTTGTCATCGACGGAA-1', '10mix-ICI2_TTTGTCATCTATCGCC-1'],
      dtype='object', length=21580)

In [12]:
import re

# Step 1: extract clean barcode
barcodes = (
    adata_scvelo.obs_names
    .str.split(":").str[1]      # take part after :
    .str.replace("x", "", regex=False)  # remove trailing x
)

# Step 2: clean sample names
samples = (
    adata_scvelo.obs["sample_id"]
    .str.replace("2021_", "", regex=False)
)

# Step 3: build matching obs_names
new_names = samples + "_" + barcodes + "-1"

adata_scvelo.obs_names = new_names


In [13]:
adata_scvelo.obs

,sample_id,batch
10mix-ICI1_AAAGATGGTGATGCCC-1,2021_10mix-ICI1,2021_10mix-ICI1
10mix-ICI1_AAACCTGAGGGCACTA-1,2021_10mix-ICI1,2021_10mix-ICI1
10mix-ICI1_AAACGGGAGAGCTGCA-1,2021_10mix-ICI1,2021_10mix-ICI1
10mix-ICI1_AAAGATGCAGTAGAGC-1,2021_10mix-ICI1,2021_10mix-ICI1
10mix-ICI1_AAACCTGCAGTTCCCT-1,2021_10mix-ICI1,2021_10mix-ICI1
...,...,...
GF-ICI2_TTTGCGCGTCATACTG-1,2021_GF-ICI2,2021_GF-ICI2
GF-ICI2_TTTGGTTAGACAGGCT-1,2021_GF-ICI2,2021_GF-ICI2
GF-ICI2_TTTGGTTTCTTGACGA-1,2021_GF-ICI2,2021_GF-ICI2
GF-ICI2_TTTGTCATCTGACCTC-1,2021_GF-ICI2,2021_GF-ICI2


In [14]:
adata.obs_names

Index(['10mix-ICI1_AAACCTGAGAGCCTAG-1', '10mix-ICI1_AAACCTGAGATGTGGC-1',
       '10mix-ICI1_AAACCTGAGCCACGTC-1', '10mix-ICI1_AAACCTGAGGCTCTTA-1',
       '10mix-ICI1_AAACCTGAGGGCACTA-1', '10mix-ICI1_AAACCTGAGGGCTTGA-1',
       '10mix-ICI1_AAACCTGAGGTTACCT-1', '10mix-ICI1_AAACCTGAGTCCAGGA-1',
       '10mix-ICI1_AAACCTGCAAAGCAAT-1', '10mix-ICI1_AAACCTGCAACCGCCA-1',
       ...
       '10mix-ICI2_TTTGTCACATTGTGCA-1', '10mix-ICI2_TTTGTCAGTACCGTTA-1',
       '10mix-ICI2_TTTGTCAGTCGAGATG-1', '10mix-ICI2_TTTGTCAGTGCGATAG-1',
       '10mix-ICI2_TTTGTCAGTGTTTGTG-1', '10mix-ICI2_TTTGTCATCACGAAGG-1',
       '10mix-ICI2_TTTGTCATCAGCTCGG-1', '10mix-ICI2_TTTGTCATCCATGCTC-1',
       '10mix-ICI2_TTTGTCATCGACGGAA-1', '10mix-ICI2_TTTGTCATCTATCGCC-1'],
      dtype='object', length=21580)

In [15]:
len(set(adata.obs_names)), len(set(adata_scvelo.obs_names)), len(
    set(adata.obs_names) & set(adata_scvelo.obs_names)
)

(21580, 72592, 21580)

In [16]:
adata_scvelo = scv.utils.merge(adata_scvelo, adata)

In [17]:
scv.pp.filter_and_normalize(adata_scvelo)

Normalized count data: X, spliced, unspliced.


In [18]:
# 1. Get the shared cell barcodes in the exact order of your scVelo object
scvelo_cells = adata_scvelo.obs_names

# 2. Extract the integer positional indices of these cells from the main adata object
cell_indices = adata.obs_names.get_indexer(scvelo_cells)

# 3. Slice the connectivities and distances matrices across both dimensions (rows and columns)
adata_scvelo.obsp['connectivities'] = adata.obsp['connectivities'][cell_indices, :][:, cell_indices]
adata_scvelo.obsp['distances'] = adata.obsp['distances'][cell_indices, :][:, cell_indices]

# 4. (Optional) Verify the slot update
print(adata_scvelo)

AnnData object with n_obs × n_vars = 21580 × 17581
    obs: 'sample_id', 'batch', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'adata_name', 'condition', 'batch_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'rna_leiden', 'rna_leiden_05', 'gex03', 'group4', 'cell_annotation_05', 'n_counts'
    var: 'gene_ids', 'feature_types', 'mito', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'n_cells', 'gene_count_corr'
    uns: 'cell_annotation_05_colors', 'gex03', 'gex03_colors', 'group4_colors', 'hvg', 

In [19]:
adata_scvelo

AnnData object with n_obs × n_vars = 21580 × 17581
    obs: 'sample_id', 'batch', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'adata_name', 'condition', 'batch_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'rna_leiden', 'rna_leiden_05', 'gex03', 'group4', 'cell_annotation_05', 'n_counts'
    var: 'gene_ids', 'feature_types', 'mito', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'n_cells', 'gene_count_corr'
    uns: 'cell_annotation_05_colors', 'gex03', 'gex03_colors', 'group4_colors', 'hvg', 

In [20]:
mask = (adata_scvelo.obs["cell_annotation_05"] != "Cycling") & (
    adata_scvelo.obs["group4"] == "ctrl"
)

# 2. Subset the AnnData object
without_cycling_ctrl= adata_scvelo[mask]

In [21]:
adata_scvelo = without_cycling_ctrl

In [22]:
adata_scvelo.X = adata_scvelo.layers['spliced'].copy()

In [23]:
results = sccellfie.run_sccellfie_pipeline(adata_scvelo,
                                           organism='mouse',
                                           sccellfie_data_folder=None,
                                           n_counts_col=None, # Total counts per cell will be computed if left as None,
                                           process_by_group=False, # Whether to do the processing by cell groups
                                           groupby=None, # Column indicating cell groups if `process_by_group=True`
                                           neighbors_key='neighbors', # Neighbors information if precomputed. Otherwise, it will be computed here
                                           n_neighbors=10, # Number of neighbors to use
                                           batch_key=None, # there is no batch_key in this dataset
                                           threshold_key='sccellfie_threshold',  # This is for using the default database. If personalized thresholds are used, specificy column name
                                           smooth_cells=True, # Whether to perform gene expression smoothing before running the tool
                                           alpha=0.33, # Importance of neighbors' expression for the smoothing (0 to 1)
                                           chunk_size=5000, # Number of chunks to run the processing steps (helps with the memory)
                                           disable_pbar=False,
                                           save_folder=None, # In case results will be saved. If so, results will not be returned and should be loaded from the folder (see sccellfie.io.load_data function
                                           save_filename=None # Name for saving the files, otherwise a default name will be used
                                          )


==== scCellFie Pipeline: Initializing ====
Loading scCellFie database for organism: mouse

==== scCellFie Pipeline: Processing entire dataset ====

---- scCellFie Step: Preprocessing data ----

---- scCellFie Step: Preparing inputs ----
Gene names corrected to match database: 9
Shape of new adata object: (21580, 683)
Number of GPRs: 623
Shape of tasks by genes: (193, 683)
Shape of reactions by genes: (623, 683)
Shape of tasks by reactions: (193, 623)

---- scCellFie Step: Smoothing gene expression ----


Smoothing Expression: 100%|██████████| 1/1 [00:00<00:00, 10.81it/s]


---- scCellFie Step: Computing gene scores ----



---- scCellFie Step: Computing reaction activity ----


Cell Rxn Activities: 100%|██████████| 21580/21580 [02:29<00:00, 144.23it/s]



---- scCellFie Step: Computing metabolic task activity ----
Removed 9 metabolic tasks with zeros across all cells.

==== scCellFie Pipeline: Processing completed successfully ====


In [24]:
results.keys()

dict_keys(['adata', 'gpr_rules', 'task_by_gene', 'rxn_by_gene', 'task_by_rxn', 'rxn_info', 'task_info', 'thresholds', 'organism'])

In [25]:
sccellfie.io.save_adata(adata=results['adata'], output_directory='/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/scCellFie', filename='Honda_tumor_scCellFie_without_cycling_ctrl')

/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/scCellFie/Honda_tumor_scCellFie_without_cycling_ctrl.h5ad was correctly saved
/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/scCellFie/Honda_tumor_scCellFie_without_cycling_ctrl_reactions.h5ad was correctly saved
/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/scCellFie/Honda_tumor_scCellFie_without_cycling_ctrl_metabolic_tasks.h5ad was correctly saved
